In [1]:
# instal libs

!pip install langchain langchain-community langchain-huggingface
!pip install faiss-cpu
!pip install pypdf
!pip install sentence-transformers
!pip install groq langchain-groq
!pip install langchain-text-splitters
!pip install streamlit pyngrok

print("ok 1")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.8 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.3/346.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 63.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

In [2]:
import os
import glob

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq



print("ok 2")

/tmp/ipykernel_978/982684617.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


ok 2


In [3]:
# download 7 papers to collab


import os
import urllib.request

os.makedirs('papers', exist_ok=True)

papers = {
    "attention_is_all_you_need.pdf": "https://arxiv.org/pdf/1706.03762.pdf",
    "bert.pdf": "https://arxiv.org/pdf/1810.04805.pdf",
    "gpt3.pdf": "https://arxiv.org/pdf/2005.14165.pdf",
    "rag.pdf": "https://arxiv.org/pdf/2005.11401.pdf",
    "sentence_bert.pdf": "https://arxiv.org/pdf/1908.10084.pdf",
    "lora.pdf": "https://arxiv.org/pdf/2106.09685.pdf",
    "llama2.pdf": "https://arxiv.org/pdf/2307.09288.pdf"
}

for filename, url in papers.items():
    filepath = f"papers/{filename}"
    if os.path.exists(filepath):
        print(f"Already exists: {filename}")
    else:
        print(f"Downloading: {filename}...")
        urllib.request.urlretrieve(url, filepath)
        print(f"Done: {filename}")

print(f"Files in papers/: {os.listdir('papers')}")

print("ok 3")

Downloading: attention_is_all_you_need.pdf...
Done: attention_is_all_you_need.pdf
Downloading: bert.pdf...
Done: bert.pdf
Downloading: gpt3.pdf...
Done: gpt3.pdf
Downloading: rag.pdf...
Done: rag.pdf
Downloading: sentence_bert.pdf...
Done: sentence_bert.pdf
Downloading: lora.pdf...
Done: lora.pdf
Downloading: llama2.pdf...
Done: llama2.pdf
Files in papers/: ['sentence_bert.pdf', 'bert.pdf', 'lora.pdf', 'attention_is_all_you_need.pdf', 'rag.pdf', 'gpt3.pdf', 'llama2.pdf']
ok 3


In [4]:
# load the content

from langchain_community.document_loaders import PyPDFLoader
import glob

pdf_files = glob.glob("papers/*.pdf")
for f in pdf_files:
    print(f"  - {f}")

all_documents = []

for pdf_path in pdf_files:
    print(f"\nLoading: {pdf_path}")

    loader = PyPDFLoader(pdf_path)


    pages = loader.load()
    all_documents.extend(pages)

    print(f"  Loaded {len(pages)} pages")
    print(f"  Preview: {pages[0].page_content[:200]}...")

print(f"\nTotal pages across all papers: {len(all_documents)}")
print("\nSample document metadata:", all_documents[0].metadata)

print("ok 4")

  - papers/sentence_bert.pdf
  - papers/bert.pdf
  - papers/lora.pdf
  - papers/attention_is_all_you_need.pdf
  - papers/rag.pdf
  - papers/gpt3.pdf
  - papers/llama2.pdf

Loading: papers/sentence_bert.pdf
  Loaded 11 pages
  Preview: Sentence-BERT: Sentence Embeddings using Siamese BERT-Networks
Nils Reimers and Iryna Gurevych
Ubiquitous Knowledge Processing Lab (UKP-TUDA)
Department of Computer Science, Technische Universit¨at Da...

Loading: papers/bert.pdf
  Loaded 16 pages
  Preview: BERT: Pre-training of Deep Bidirectional Transformers for
Language Understanding
Jacob Devlin Ming-Wei Chang Kenton Lee Kristina Toutanova
Google AI Language
{jacobdevlin,mingweichang,kentonl,kristout...

Loading: papers/lora.pdf
  Loaded 26 pages
  Preview: LORA: L OW-R ANK ADAPTATION OF LARGE LAN-
GUAGE MODELS
Edward Hu∗ Yelong Shen∗ Phillip Wallis Zeyuan Allen-Zhu
Yuanzhi Li Shean Wang Lu Wang Weizhu Chen
Microsoft Corporation
{edwardhu, yeshe, phwalli...

Loading: papers/attention_is_all_you_need.

In [5]:
# spliting the document into chunks

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
     separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_documents(all_documents)
print(f"Original pages: {len(all_documents)}")
print(f"Chunks after splitting: {len(chunks)}")
print(f"Average chunks per page: {len(chunks)/len(all_documents):.1f}")
print(f"\nSample chunk content:")
print(f"  Text: {chunks[5].page_content[:300]}...")
print(f"  Metadata: {chunks[5].metadata}")
print(f"  Length: {len(chunks[5].page_content)} characters")

print("ok 5")

Original pages: 239
Chunks after splitting: 1057
Average chunks per page: 4.4

Sample chunk content:
  Text: most similar sentence pair in a collection of 10,000
sentences is reduced from 65 hours with BERT to
the computation of 10,000 sentence embeddings
(~5 seconds with SBERT) and computing cosine-
similarity (~0.01 seconds). By using optimized
index structures, ﬁnding the most similar Quora
question can...
  Metadata: {'producer': 'pdfTeX-1.40.17', 'creator': 'LaTeX with hyperref package', 'creationdate': '2019-08-28T00:45:27+00:00', 'author': '', 'keywords': '', 'moddate': '2019-08-28T00:45:27+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'papers/sentence_bert.pdf', 'total_pages': 11, 'page': 1, 'page_label': '2'}
  Length: 974 characters
ok 5


In [6]:
# creating embedding and build vector

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
   )


vectorstore = FAISS.from_documents(
    chunks,
    embedding_model
)

vectorstore.save_local("faiss_index")
print(f"\nVector database created with {len(chunks)} vectors!")
print("Saved to faiss_index/")
print("Each vector has 384 dimensions")


test_query = "What is attention mechanism?"
results = vectorstore.similarity_search(test_query, k=2)
print(f"\nTest search: '{test_query}'")
print(f"Top result preview: {results[0].page_content[:200]}...")
print(f"From paper: {results[0].metadata.get('source', 'unknown')}")

print("ok 6")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Vector database created with 1057 vectors!
Saved to faiss_index/
Each vector has 384 dimensions

Test search: 'What is attention mechanism?'
Top result preview: 3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as ...
From paper: papers/attention_is_all_you_need.pdf
ok 6


In [7]:
# Setting up the QA chain

os.environ["GROQ_API_KEY"] = "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"

# Initialize the LLM
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0,
)

# load vector store
vectorstore = FAISS.load_local(
    "faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

# create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

# Shoot queations

def ask_rag(question):
    print(f"\n{'='*60}")
    print(f"QUESTION: {question}")
    print('='*60)


    relevant_chunks = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in relevant_chunks])
    prompt = f"""You are a helpful research assistant analyzing ML research papers.
Use the context below to answer the question as best you can.
If the context contains partial information, use it to give the best possible answer.
Always mention which paper the information comes from.

Context:
{context}

Question: {question}

Answer:"""


    response = llm.invoke(prompt)
    answer = response.content

    print(f"\nANSWER:\n{answer}")

    print(f"\nSOURCES USED:")
    seen = set()
    for doc in relevant_chunks:
        source = doc.metadata.get('source', 'unknown').replace('papers/', '')
        page = doc.metadata.get('page', 0) + 1
        key = f"{source}_{page}"
        if key not in seen:
            seen.add(key)
            print(f"  📄 {source} — Page {page}")

    return answer, relevant_chunks

# Test it

ask_rag("What do BERT and GPT have in common?")
ask_rag("How does LoRA reduce training cost?")
ask_rag("Which models use the Transformer architecture?")


QUESTION: What do BERT and GPT have in common?

ANSWER:
According to the paper, BERT and GPT have the following in common:

1. Both are pre-training methods for natural language processing (NLP) tasks.
2. Both are trained on a large text corpus: BERT is trained on the BooksCorpus (800M words) and Wikipedia (2,500M words), while GPT is trained on the BooksCorpus (800M words).
3. Both use a Transformer-based architecture, although BERT uses a bidirectional Transformer and GPT uses a left-to-right Transformer.
4. Both are fine-tuning approaches, meaning they are trained on a specific task after pre-training on a large corpus.

The paper also mentions that many of the design decisions in BERT were intentionally made to make it as close to GPT as possible so that the two methods could be minimally compared.

SOURCES USED:
  📄 bert.pdf — Page 14
  📄 bert.pdf — Page 13
  📄 bert.pdf — Page 3
  📄 gpt3.pdf — Page 20
  📄 gpt3.pdf — Page 18
  📄 bert.pdf — Page 6

QUESTION: How does LoRA reduce tr

('The following models use the Transformer architecture:\n\n1. Transformer (Vaswani et al., 2017) - a sequence-to-sequence architecture that makes heavy use of self-attention.\n2. BERT (Devlin et al., 2019b) - a large Transformer language model.\n3. GPT-2 (Radford et al., b) - a large Transformer language model.\n\nAdditionally, the paper mentions that the BERT model is based on the original Transformer implementation described in Vaswani et al. (2017) and released in the tensor2tensor library.\n\nReferences:\n- Vaswani et al. (2017)\n- Devlin et al. (2019b)\n- Radford et al. (b)',
 [Document(id='2b2bf462-83a1-43b5-84e6-41ca9be1ff17', metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-10-19T00:57:13+00:00', 'author': '', 'keywords': '', 'moddate': '2021-10-19T00:57:13+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '